# Per-Stage LoRA Adapters v3 — Stage 13 (entity validation) + Stage 12/10.5 (relation-only)

**Two independent adapters, trained back-to-back overnight, in one Run All.** Neither task
shares training data with the other, and neither is mixed with counting/typing/reading —
that mixing is exactly what v1/v2 showed damages unrelated skills. This notebook trains:

1. **Stage 13 adapter** (NEW) — keep/remove entity validation, including deliberate absence
   examples (near-miss shifted boxes, wrong-size boxes, empty-region decoys). Nobody passes
   this stage today (base 50% = chance, v2 adapter 35% = below chance with a "yes there's a
   symbol" bias). Highest-value adapter to build — it's the one stage nothing currently works on.
2. **Stage 12/10.5 adapter** (v3, relation-ONLY) — same connectivity task as v2, but: fresh
   LoRA, ~3x the training pairs (adds the previously-unused `Patched/Dataset PID` tree,
   schema-verified before use — falls back to OPEN100-only if it doesn't match), 5 phrasing/
   format variants instead of 1. Question this answers: does a dedicated, uncontaminated,
   bigger-data adapter beat prompted-base's 80% (v2 scored 84% at n=25 — noise, not a real
   margin)? **Answer is not yet known — that's what tonight's run + tomorrow's eval decides.**

**Both use language-modules-only LoRA** (vision tower excluded from `target_modules`) — the
leading suspect for why v1/v2 destroyed reading was `all-linear` LoRA rewriting the vision
encoder from gradients that never rewarded visual precision. Target modules are discovered
at runtime by inspecting the loaded model (printed for a sanity check), not hardcoded.

**Overnight resilience, sequencing included:** each phase checkpoints every 200 steps to its
own HF path and auto-resumes mid-epoch exactly like the v2 notebook. On top of that, this
notebook checks HF for **which phases have already fully completed** (their last epoch's
snapshot existing) and automatically:
- skips any phase that's done,
- resumes any phase that's partially done from its last checkpoint,
- and runs the next phase after the current one finishes —
all inside one **Run All**, from any fresh kernel, without you needing to know or choose
which phase died. If the kernel dies at 3am, reopen tomorrow, Run All, done.

## 1. Config

In [ ]:
HF_TOKEN = "paste-your-hf-token-here"

DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# Phase A: Stage 13 entity validation
STAGE13_CKPT_SUBDIR = "v3-stage13"
STAGE13_N_EPOCHS = 3
STAGE13_MAX_PER_SHEET = 12   # keep + remove examples per Gupta sheet (72 sheets -> ~864 max)

# Phase B: Stage 12/10.5 relation-only (bigger, uncontaminated, no other tasks mixed in)
RELATION_CKPT_SUBDIR = "v3-relation"
RELATION_N_EPOCHS = 3
RELATION_MAX_PATCHES_OPEN100 = 1600     # up to all of Patched/PID2Graph OPEN100
RELATION_MAX_PATCHES_DATASETPID = 4000  # NEW tree, schema-verified before use; cap for time
RELATION_PAIRS_PER_PATCH = 12

CHECKPOINT_EVERY_N_STEPS = 200
LR = 1e-4
TIME_BUDGET_HOURS_PER_PHASE = 8   # each phase gets its own budget; stops cleanly, resumes next run

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here", "paste your HF token"

## 2. Install + GPU check

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q peft pytesseract huggingface_hub
!pip uninstall -y torchao -q

import torch
assert torch.cuda.is_available(), "No GPU - set Runtime type first"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 3. Data from HF -> local disk

In [ ]:
import zipfile, time, json, random
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi, snapshot_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)
t0 = time.time()
for fname, extract_to in [
    ("gupta_pid/PID_Dataset.zip", DATA / "gupta"),
    ("pid2graph/PID2Graph.zip", DATA / "pid2graph"),
]:
    if extract_to.exists() and any(extract_to.iterdir()):
        print(f"{extract_to} already extracted"); continue
    zp = hf_hub_download(repo_id=DATA_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(extract_to)
    print(f"extracted {fname} ({time.time()-t0:.0f}s)")

GUPTA_RAW = DATA / "gupta" / "PID_Dataset" / "0__raw_data"
P2G_PATCHED = DATA / "pid2graph" / "PID2Graph" / "Patched"
OPEN100_DIR = P2G_PATCHED / "PID2Graph OPEN100"
DATASETPID_DIR = P2G_PATCHED / "Dataset PID"
for p in [GUPTA_RAW / "sheets" / "train", OPEN100_DIR]:
    assert p.exists(), f"missing: {p}"
print("data ready. Dataset PID tree present:", DATASETPID_DIR.exists())

## 4. Model + language-only LoRA target-module discovery

Discovers `target_modules` at runtime by inspecting the loaded model's Linear layers and
excluding anything under the vision tower (name containing "visual"/"vision"/"image"). This
avoids hardcoding module names that could be wrong for this model version. Prints what's
included/excluded so you can sanity-check before training starts.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
import torch.nn as nn

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
base_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda").eval()
print("Base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

_EXCLUDE_KW = ["visual", "vision", "image_tower"]
_leaf_names, _excluded_sample = set(), []
for name, module in base_model.named_modules():
    if isinstance(module, nn.Linear):
        if any(kw in name.lower() for kw in _EXCLUDE_KW):
            if len(_excluded_sample) < 5:
                _excluded_sample.append(name)
            continue
        _leaf_names.add(name.split(".")[-1])

LANGUAGE_ONLY_TARGET_MODULES = sorted(_leaf_names)
print("Language-only target modules (LoRA will touch these):", LANGUAGE_ONLY_TARGET_MODULES)
print("Sample of EXCLUDED (vision tower) module paths:", _excluded_sample)
assert LANGUAGE_ONLY_TARGET_MODULES, "discovery found nothing - check exclude keywords match this model version"

## 5. Checkpoint helpers (shared by both phases)

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=CKPT_REPO, repo_type="model", private=True, exist_ok=True)

def phase_epoch_exists(ckpt_subdir, epoch):
    """True if that epoch's full snapshot is already on HF (phase-completion signal)."""
    try:
        files = api.list_repo_files(repo_id=CKPT_REPO, repo_type="model")
        return any(f.startswith(f"{ckpt_subdir}/epoch_{epoch}/") for f in files)
    except Exception:
        return False

def load_fresh_lora(base_model):
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                     target_modules=LANGUAGE_ONLY_TARGET_MODULES, task_type="CAUSAL_LM")
    m = get_peft_model(base_model, cfg)
    m.print_trainable_parameters()
    return m

def resume_or_fresh(model, ckpt_subdir):
    """Pulls latest/ checkpoint for this phase if present; returns (start_epoch, start_step)."""
    local = Path(f"/content/ckpt_{ckpt_subdir.replace('/', '_')}")
    latest_dir = local / "latest"
    progress_file = latest_dir / "progress.json"
    try:
        snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                          allow_patterns=[f"{ckpt_subdir}/latest/*"], local_dir=str(local),
                          local_dir_use_symlinks=False)
        # snapshot_download preserves the repo path prefix; locate the actual files
        pulled = local / ckpt_subdir / "latest"
        if pulled.exists() and pulled != latest_dir:
            latest_dir = pulled
            progress_file = latest_dir / "progress.json"
    except Exception as e:
        print(f"  no remote snapshot for {ckpt_subdir} ({type(e).__name__})")
    if progress_file.exists() and (latest_dir / "adapter_model.safetensors").exists():
        progress = json.loads(progress_file.read_text())
        model.load_adapter(str(latest_dir), adapter_name="default", is_trainable=True)
        ep, st = progress["epoch"], progress["step"] + 1
        print(f"RESUMED {ckpt_subdir}: epoch {ep}, step {st}")
        return progress["epoch"], progress["step"] + 1, latest_dir, progress_file
    print(f"No checkpoint for {ckpt_subdir} - fresh start")
    return 0, 0, latest_dir, progress_file

def save_and_push(model, ckpt_subdir, latest_dir, progress_file, epoch, step, also_epoch=False):
    latest_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(latest_dir))
    progress_file.write_text(json.dumps({"epoch": epoch, "step": step}))
    try:
        api.upload_folder(folder_path=str(latest_dir), path_in_repo=f"{ckpt_subdir}/latest",
                          repo_id=CKPT_REPO, repo_type="model")
        if also_epoch:
            api.upload_folder(folder_path=str(latest_dir), path_in_repo=f"{ckpt_subdir}/epoch_{epoch}",
                              repo_id=CKPT_REPO, repo_type="model")
    except Exception as e:
        print(f"  [warn] HF push failed ({type(e).__name__}) - local checkpoint kept, retries next interval")

def build_labeled_inputs(processor, model, image, prompt, target_text):
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": prompt}]}]
    prompt_inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    prompt_len = prompt_inputs["input_ids"].shape[1]
    full = processor.apply_chat_template(
        messages + [{"role": "assistant", "content": [{"type": "text", "text": target_text}]}],
        tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")
    labels = full["input_ids"].clone()
    labels[:, :prompt_len] = -100
    full["labels"] = labels
    return {k: v.to(model.device) for k, v in full.items()}

def run_training_phase(model, examples, ckpt_subdir, n_epochs, time_budget_hours):
    """Generic resumable training loop, used for both Stage 13 and relation phases."""
    start_epoch, start_step, latest_dir, progress_file = resume_or_fresh(model, ckpt_subdir)
    if phase_epoch_exists(ckpt_subdir, n_epochs - 1):
        print(f"=== {ckpt_subdir} already fully trained ({n_epochs} epochs) - skipping ===")
        return True

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    run_t0 = time.time()
    budget_s = time_budget_hours * 3600
    stopped_for_time = False

    for epoch in range(start_epoch, n_epochs):
        random.seed(epoch)
        order = list(range(len(examples)))
        random.shuffle(order)
        skip_to = start_step if epoch == start_epoch else 0
        losses = []
        t0 = time.time()
        for step, idx in enumerate(order):
            if step < skip_to:
                continue
            if time.time() - run_t0 > budget_s:
                save_and_push(model, ckpt_subdir, latest_dir, progress_file, epoch, step - 1)
                print(f"TIME BUDGET REACHED for {ckpt_subdir} at epoch {epoch} step {step-1}. "
                      "Rerun the notebook to continue.")
                stopped_for_time = True
                break
            ex = examples[idx]
            try:
                inputs = build_labeled_inputs(processor, model, ex["image"], ex["prompt"], ex["target_text"])
                out = model(**inputs)
                loss = out.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                losses.append(loss.item())
            except torch.cuda.OutOfMemoryError:
                optimizer.zero_grad(set_to_none=True)
                import gc; gc.collect(); torch.cuda.empty_cache()
                print(f"  [skip] {ckpt_subdir} step {step} OOM - cleared, continuing")
                continue
            except Exception as e:
                print(f"  [skip] {ckpt_subdir} step {step}: {type(e).__name__}: {e}")
                continue
            if step % 50 == 0:
                recent = losses[-50:]
                avg = sum(recent) / len(recent) if recent else float("nan")
                print(f"{ckpt_subdir} epoch {epoch} step {step}/{len(order)} "
                      f"avg_loss={avg:.4f} elapsed={time.time()-t0:.0f}s")
            if step % CHECKPOINT_EVERY_N_STEPS == 0 and step > 0:
                save_and_push(model, ckpt_subdir, latest_dir, progress_file, epoch, step)
                print(f"  [checkpoint] {ckpt_subdir} epoch {epoch} step {step}")
        if stopped_for_time:
            return False
        save_and_push(model, ckpt_subdir, latest_dir, progress_file, epoch, len(order) - 1, also_epoch=True)
        print(f"=== {ckpt_subdir} epoch {epoch} done (pushed epoch_{epoch}/) ===")
        if losses:
            print(f"  mean_loss={sum(losses)/len(losses):.4f}")
        start_step = 0
    print(f"=== {ckpt_subdir} training complete ({n_epochs} epochs) ===")
    return True

## 6. Build Stage 13 training examples (keep/remove, WITH absence supervision)

Balanced 50/50 real-vs-absent, three absence types (empty region, shifted near-miss, wrong
size) so the model can't learn "always yes" the way the v2 adapter did. Multiple phrasings
and BOTH keep/remove and yes/no answer formats, so it isn't locked to one template.

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
rng13 = random.Random(13)

def load_boxes(label_path, W, H):
    boxes = []
    for line in Path(label_path).read_text().splitlines():
        if line.strip():
            parts = line.split()
            cx, cy, w, h = (float(v) for v in parts[1:5])
            boxes.append([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H])
    return boxes

def overlaps_any(box, boxes):
    return any(box[0] < b[2] and box[2] > b[0] and box[1] < b[3] and box[3] > b[1] for b in boxes)

S13_PROMPTS = [
    ("A symbol detector claims there is a P&ID equipment symbol at [{cb}] in this crop "
     "(pixel coords, top-left origin). Verify against the pixels: is there really a discrete "
     "equipment symbol there (valve, instrument, fitting)? Answer keep or remove.",
     {"real": "keep", "absent": "remove"}),
    ("Does the region marked [{cb}] in this crop actually contain a P&ID equipment symbol? "
     "Answer yes or no.",
     {"real": "yes", "absent": "no"}),
    ("A detection at [{cb}] is proposed for this P&ID crop. Confirm or reject it: is a real "
     "symbol present at that location? Answer confirm or reject.",
     {"real": "confirm", "absent": "reject"}),
]

def build_stage13_examples(max_per_sheet):
    examples = []
    sheets = sorted((GUPTA_RAW / "sheets" / "train").glob("*.*"))
    for sheet_path in sheets:
        img = Image.open(sheet_path).convert("RGB")
        W, H = img.size
        boxes = load_boxes(GUPTA_RAW / "labels" / "train" / f"{sheet_path.stem}.txt", W, H)
        if not boxes:
            continue
        n_here = 0
        rng13.shuffle(boxes)
        for real_box in boxes:
            if n_here >= max_per_sheet:
                break
            # positive: real box, tight margin
            m = 60
            crop_box = [max(0, real_box[0]-m), max(0, real_box[1]-m),
                       min(W, real_box[2]+m), min(H, real_box[3]+m)]
            cb = [int(real_box[0]-crop_box[0]), int(real_box[1]-crop_box[1]),
                  int(real_box[2]-crop_box[0]), int(real_box[3]-crop_box[1])]
            crop = img.crop([int(v) for v in crop_box])
            template, answers = rng13.choice(S13_PROMPTS)
            examples.append({"image": crop, "prompt": template.format(cb=cb),
                             "target_text": answers["real"]})
            n_here += 1

            # negative: pick one of three absence types
            absence_type = rng13.choice(["empty", "shifted", "wrong_size"])
            bad_box = None
            if absence_type == "empty":
                for _ in range(80):
                    size = rng13.uniform(30, 80)
                    x0 = rng13.uniform(0, W-size); y0 = rng13.uniform(0, H-size)
                    cand = [x0, y0, x0+size, y0+size]
                    if not overlaps_any(cand, boxes):
                        bad_box = cand; break
            elif absence_type == "shifted":
                bw, bh = real_box[2]-real_box[0], real_box[3]-real_box[1]
                dx = rng13.choice([-1, 1]) * bw * rng13.uniform(1.2, 2.0)
                dy = rng13.choice([-1, 1]) * bh * rng13.uniform(1.2, 2.0)
                cand = [real_box[0]+dx, real_box[1]+dy, real_box[2]+dx, real_box[3]+dy]
                if 0 <= cand[0] and cand[2] <= W and 0 <= cand[1] and cand[3] <= H \
                        and not overlaps_any(cand, boxes):
                    bad_box = cand
            else:  # wrong_size: real center, much bigger box (engulfs empty space)
                cx, cy = (real_box[0]+real_box[2])/2, (real_box[1]+real_box[3])/2
                scale = rng13.uniform(3.0, 5.0)
                bw = (real_box[2]-real_box[0]) * scale
                bh = (real_box[3]-real_box[1]) * scale
                cand = [cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2]
                if cand[0] >= 0 and cand[2] <= W and cand[1] >= 0 and cand[3] <= H:
                    bad_box = cand
            if bad_box is None:
                continue
            m = 60
            crop_box = [max(0, bad_box[0]-m), max(0, bad_box[1]-m),
                       min(W, bad_box[2]+m), min(H, bad_box[3]+m)]
            cb = [int(bad_box[0]-crop_box[0]), int(bad_box[1]-crop_box[1]),
                  int(bad_box[2]-crop_box[0]), int(bad_box[3]-crop_box[1])]
            crop = img.crop([int(v) for v in crop_box])
            template, answers = rng13.choice(S13_PROMPTS)
            examples.append({"image": crop, "prompt": template.format(cb=cb),
                             "target_text": answers["absent"]})
            n_here += 1
    return examples

stage13_examples = build_stage13_examples(STAGE13_MAX_PER_SHEET)
random.Random(0).shuffle(stage13_examples)
print(f"Stage 13 pool: {len(stage13_examples)} examples "
      f"(target ~50/50 keep vs remove; absence types: empty/shifted/wrong_size)")

## 7. Run Phase A — Stage 13 (skips automatically if already complete on HF)

In [ ]:
model = load_fresh_lora(base_model)
run_training_phase(model, stage13_examples, STAGE13_CKPT_SUBDIR,
                   STAGE13_N_EPOCHS, TIME_BUDGET_HOURS_PER_PHASE)

## 8. Build relation-only training examples (bigger, uncontaminated, varied)

Reuses the verified graphml parser. Adds the previously-unused `Patched/Dataset PID` tree,
but only after confirming its schema matches (same xmin/ymin/xmax/ymax + edge structure) —
falls back to OPEN100-only if a sample fails to parse the expected way.

In [ ]:
import xml.etree.ElementTree as ET
rng_rel = random.Random(777)

def parse_graphml(path):
    root = ET.parse(path).getroot()
    ns = {"g": "http://graphml.graphdrawing.org/xmlns"}
    keymap = {k.get("id"): k.get("attr.name") for k in root.findall("g:key", ns)}
    nodes, edges = {}, set()
    for node in root.iter("{http://graphml.graphdrawing.org/xmlns}node"):
        vals = {keymap.get(d.get("key"), ""): d.text for d in node.findall("g:data", ns)}
        try:
            nodes[node.get("id")] = [float(vals["xmin"]), float(vals["ymin"]),
                                     float(vals["xmax"]), float(vals["ymax"])]
        except (KeyError, TypeError, ValueError):
            continue
    for e in root.iter("{http://graphml.graphdrawing.org/xmlns}edge"):
        if e.get("source") in nodes and e.get("target") in nodes:
            edges.add(frozenset((e.get("source"), e.get("target"))))
    return nodes, edges

def schema_check(directory, n_samples=5):
    """Sample a few graphml files; require most to parse with usable nodes+edges."""
    samples = list(directory.rglob("*.graphml"))[:n_samples]
    if not samples:
        return False
    ok = 0
    for gml in samples:
        try:
            nodes, edges = parse_graphml(gml)
            if len(nodes) >= 2:
                ok += 1
        except ET.ParseError:
            continue
    return ok >= max(1, n_samples // 2)

REL_PROMPTS = [
    ("In this P&ID crop there is a symbol at [{a}] and another at [{b}] "
     "(pixel coordinates, top-left origin). Are these two symbols directly connected "
     "to each other? Answer yes or no.",
     {"pos": "Yes - these two symbols are directly connected.",
      "neg": "No - these two symbols are not directly connected."}),
    ("Two symbols are marked in this P&ID crop: one at [{a}], another at [{b}]. "
     "Is there a direct connection (pipe or signal line) between them? "
     "Answer true or false.",
     {"pos": "True - there is a direct connection between them.",
      "neg": "False - there is no direct connection between them."}),
    ("Looking at the crop, symbol A is at [{a}] and symbol B is at [{b}]. "
     "Are A and B connected to each other directly? Answer connected or not connected.",
     {"pos": "Connected - A and B are directly connected.",
      "neg": "Not connected - A and B are not directly connected."}),
    ("This crop shows two P&ID symbols, at [{a}] and [{b}]. Confirm: do these symbols "
     "share a direct line connection? Answer confirm or deny.",
     {"pos": "Confirm - they share a direct connection.",
      "neg": "Deny - they do not share a direct connection."}),
    ("Symbol positions in this P&ID crop: [{a}] and [{b}]. Is there a pipe or signal "
     "line directly joining them? Answer yes or no.",
     {"pos": "Yes.", "neg": "No."}),
]

MAX_SPAN, MARGIN = 1400, 80

def build_relation_examples_from_tree(tree_dir, max_patches, pairs_per_patch, label):
    examples = []
    gmls = sorted(tree_dir.rglob("*.graphml"))
    rng_rel.shuffle(gmls)
    gmls = gmls[:max_patches]
    for gml in gmls:
        png = gml.with_suffix(".png")
        if not png.exists():
            continue
        try:
            nodes, edges = parse_graphml(gml)
        except ET.ParseError:
            continue
        if len(nodes) < 4 or not edges:
            continue
        node_ids = list(nodes)
        pos_pool = [tuple(e) for e in edges]
        rng_rel.shuffle(pos_pool)
        neg_pool, attempts = [], 0
        while len(neg_pool) < pairs_per_patch and attempts < 200:
            attempts += 1
            a, b = rng_rel.sample(node_ids, 2)
            if frozenset((a, b)) not in edges:
                neg_pool.append((a, b))
        for pair, connected in ([(p, True) for p in pos_pool[:pairs_per_patch // 2]] +
                                [(p, False) for p in neg_pool[:pairs_per_patch // 2]]):
            a, b = pair
            ba, bb = nodes[a], nodes[b]
            ux0, uy0 = min(ba[0], bb[0]), min(ba[1], bb[1])
            ux1, uy1 = max(ba[2], bb[2]), max(ba[3], bb[3])
            if ux1-ux0 > MAX_SPAN or uy1-uy0 > MAX_SPAN:
                continue
            cx0, cy0 = max(0, int(ux0-MARGIN)), max(0, int(uy0-MARGIN))
            try:
                img = Image.open(png).convert("RGB")
            except Exception:
                continue
            crop = img.crop((cx0, cy0, int(ux1+MARGIN), int(uy1+MARGIN)))
            template, answers = rng_rel.choice(REL_PROMPTS)
            a_local = [int(ba[0]-cx0), int(ba[1]-cy0), int(ba[2]-cx0), int(ba[3]-cy0)]
            b_local = [int(bb[0]-cx0), int(bb[1]-cy0), int(bb[2]-cx0), int(bb[3]-cy0)]
            prompt = template.format(a=a_local, b=b_local)
            target = answers["pos"] if connected else answers["neg"]
            examples.append({"image": crop, "prompt": prompt, "target_text": target})
    print(f"  {label}: {len(examples)} examples from {len(gmls)} patches")
    return examples

relation_examples = build_relation_examples_from_tree(
    OPEN100_DIR, RELATION_MAX_PATCHES_OPEN100, RELATION_PAIRS_PER_PATCH, "OPEN100")

if DATASETPID_DIR.exists() and schema_check(DATASETPID_DIR):
    print("Dataset PID tree schema verified - including it")
    relation_examples += build_relation_examples_from_tree(
        DATASETPID_DIR, RELATION_MAX_PATCHES_DATASETPID, RELATION_PAIRS_PER_PATCH, "Dataset PID")
else:
    print("Dataset PID tree missing or schema check failed - using OPEN100 only")

random.Random(0).shuffle(relation_examples)
print(f"\nTotal relation-only pool: {len(relation_examples)} examples (5 phrasing/format variants)")

## 9. Run Phase B — relation-only (skips automatically if already complete on HF)

In [ ]:
# Fresh LoRA for this phase too - independent adapter, not stacked on Stage 13's
model = load_fresh_lora(base_model)
run_training_phase(model, relation_examples, RELATION_CKPT_SUBDIR,
                   RELATION_N_EPOCHS, TIME_BUDGET_HOURS_PER_PHASE)

print("\n=== BOTH PHASES ATTEMPTED THIS RUN ===")
print(f"Stage 13 adapter:  {CKPT_REPO}/{STAGE13_CKPT_SUBDIR}")
print(f"Relation adapter:  {CKPT_REPO}/{RELATION_CKPT_SUBDIR}")
print("If either printed a TIME BUDGET message above, rerun this notebook (Run All) to continue -")
print("it will auto-skip whatever finished and resume/start whatever did not.")

## 10. Tomorrow — evaluate both against prompted-base (n>=100, not n=25)

Don't trust anything below n=100 given how much noise showed up at n=25 in the earlier
head-to-head. Build eval pools disjoint from training (different seed / excluded sheets),
score each adapter against the SAME prompted-base numbers (Stage 13: base="chance", relation:
base~80%), and only adopt an adapter if it beats prompted-base by a margin that survives
n>=100. This cell is a stub - fill in once tonight's training lands, reusing the
Stage4_QwenDomainBase_Eval.ipynb pattern (peft adapter load, no reload needed to compare
against base via model.disable_adapter()).

In [ ]:
print("Stub - build the n>=100 eval here once training completes.")
print("Reuse: ADAPTER_PATH_IN_REPO = 'v3-stage13/epoch_2' or 'v3-relation/epoch_2'")
print("Decision rule: adopt only if it beats prompted-base by a real margin at n>=100.")